# Concepts
In Skchange a **change detector** is composed of an **interval scorer** and a **penalty**.
A change detector is the object you use for detecting changes, an interval scorer is the user-specified component that tells the detector *what distributional feature of the data to look for changes in*,
while the penalty controls the number of detected events.
This page introduces each of these concepts to give you a high level understanding of the library's design.

## Change detectors

A change detector in Skchange is a Scikit-learn-style estimator (see [Scikit-learn compatibility](#scikit-learn-compatibility) for details) that finds *changepoints* in a single univariate or multivariate time series.
A changepoint is a point in the time series where the data distribution changes in some way.
The workflow of change detectors is familiar to anyone who has used Scikit-learn:
You first initialise the detector with an interval scorer, a penalty, and any algorithm-specific parameters, before calling `fit` on a time series to fit the detector on training data.
Finally, you can use the fitted detector to detect changepoint on new data using `predict_changepoints` or `predict`, depending on the output format you want.

`predict_changepoints(X)`
: Returns a numpy array of changepoint indices of shape `(n_changepoints,)`. A changepoint marks the *inclusive start of a new segment*. This means that when `cpts` is the array of detected changepoints, the segments of the input time series are `X[:cpts[0]], X[cpts[0]:cpts[1]], ..., X[cpts[-1]:]`.

`predict(X)`
: Returns a numpy array of segment labels of shape `(n_samples,)` — one integer label per input sample. This mirrors the output convention of Scikit-learn clusterers and classifiers, so the array plugs directly into pipelines and metrics. Note that segment labels in general can reoccur for discontiguous segments, so the labels are not necessarily ordered.

In addition to `fit`, these two methods are mandatory on all change detectors in Skchange.

As a concrete example, the snippet below generates a univariate series with a single mean change at index 20, fits `PELT` to it, and calls both methods to make the output shapes concrete.


In [ ]:
from skchange.new_api.datasets import generate_piecewise_normal_data
from skchange.new_api.detectors import PELT
from skchange.new_api.interval_scorers import L2Cost

# A univariate series with a change in mean at index 20. Reused throughout the page.
X = generate_piecewise_normal_data(means=[0, 5], lengths=[20, 10], seed=1)

detector = PELT(cost=L2Cost(), penalty=10.0).fit(X)

print("predict_changepoints:", detector.predict_changepoints(X))
print("predict:             ", detector.predict(X))


### Segment anomaly detectors
Some change detectors in Skchange can natively identify *anomalous segments* — segments where the data deviates from a "normal" or "baseline" behaviour.
In this case, the changepoints are the boundaries of the anomalous segments.
Such detectors expose the following additional method:

`predict_segment_anomalies(X)`
: Returns a numpy array of anomalous intervals of shape `(n_anomalies, 2)` with `[start, end)` rows. The samples outside these intervals are considered normal, and are all labelled as `0` in the output of `predict`.


For example, running `CAPA` on the same `X` returns the post-change region as a single anomalous segment relative to the baseline learned from the data:


In [ ]:
from skchange.new_api.detectors import CAPA
from skchange.new_api.interval_scorers import L2Saving

anomaly_detector = CAPA(segment_saving=L2Saving(), segment_penalty=10.0).fit(X)
anomaly_detector.predict_segment_anomalies(X)


### Advanced outputs
Most Skchange detectors also expose the following methods for advanced users:

`predict_scores(X, return_index=False)`
: Returns the detector's internal interval scoring objective as a 1D numpy array. The length depends on the algorithm — one entry per evaluated interval, candidate split point, etc. — and is not generally equal to `n_samples`. With `return_index=True` it returns a `(scores, index_dict)` tuple, where `index_dict` carries algorithm-specific metadata that locates each score on the input timeline. Used by the tuning module for penalty calibration.

`predict_all(X)`
: Convenience method on detectors that compute all outputs in a single pass, which could be more than the sum of the individual methods. Returns a dict whose keys are detector-specific. Useful for power users who want all the details in one go, without repeating work.


### Scikit-learn compatibility

All Skchange detectors inherit from scikit-learn's `BaseEstimator` via Skchange's `BaseChangeDetector` class, and the aim is to follow the Scikit-learn API conventions as closely as possible. This gives you sklearn-standard machinery for free, but a few sklearn tools are intentionally not supported because they assume properties that time series data do not have.

**Data types:** Like Scikit-learn estimators, Skchange detectors accept 2D array-like input of shape `(n_samples, n_features)` and return an `np.ndarray` (except the `predict_all` method intended for advanced use).

**What works:**

- **`get_params` / `set_params`** for inspecting and updating hyperparameters.
- **`sklearn.base.clone`** for making unfitted copies.
- **`Pipeline`** with sklearn transformers that don't rely on the ordering of samples.
- **Fitted-attribute convention**: attributes set in `fit` end with `_` (e.g. `detector.penalty_`), and `check_is_fitted` works as expected.

**What does not work, and why:**

- **`GridSearchCV` / `cross_val_score` and other cross-validation utilities.** Scikit-learn's cross-validation tools don't respect the ordering of samples crucial to time series, and thus cannot be used with Skchange detectors. Use the built-in tuning utilities in `skchange.new_api.tuning` instead.

## Interval scorers
The computational bottleneck of most change detection algorithms is to evaluate some kind of cost, loss, score, test statistic or similar over a large number of intervals and/or candidate split points.

Interval scorers are the main components that make Skchange modular, flexible and fast.
As mentioned above, they represent the distributional feature of the data to look for changes in — for example, the mean, the variance, regression parameters, or the full distribution.
You have already seen two examples above: `L2Cost` for detecting changes in mean, and `L2Saving` for detecting segments with anomalous means relative to a baseline.

Apart from being constructed, interval scorers are not meant to be used directly.
They are used internally by the detectors.
To make full use of the library, however, it is important to understand what they are and how they work.

An interval scorer is an abstraction that covers a wide range of components used by changepoint detection algorithms to evaluate the "goodness" of a candidate interval and/or split point(s). In Skchange, there are four types of interval scorers, each used by different detectors:

| Score type | Evaluated for | What it scores | Example detectors |
|---|---|---|---|
| Cost | start, end | The cost/loss of a model fit to a single interval `[start, end)`. | `PELT`, `CROPS` |
| Change score | start, split, end | The degree of change between two adjacent intervals `[start, split)` and `[split, end)`. A two-sample statistical test. | `SeededBinarySegmentation`, `MovingWindow` |
| Saving | start, end | The degree to which an interval `[start, end)` deviates from a fixed baseline model. A one-sample statistical test. | `CAPA` |
| Transient score | start, split1, split2, end | The degree to which an interval `[split1, split2)` is different compared to its local context `[start, split1)` and `[split2, end)`. A two-sample statistical test, but with a different splitting strategy than change scores. | `CircularBinarySegmentation` |

The computational bottleneck of most change detection algorithms is to evaluate a score over a large number of intervals. In Skchange this is solved by splitting work between `fit`/`precompute` (which prepares quantities such as cumulative sums) and `evaluate` (which uses them to score many intervals in a single, often `numba`-accelerated call).


All interval scorers share a single base class, `BaseIntervalScorer`, and a single evaluation API:

- **`fit(X, y=None) -> self`** validates the data and stores any state needed across calls.
- **`precompute(X) -> cache`** returns any precomputed quantities (for example cumulative sums) needed to evaluate many intervals quickly. Detectors call this internally.
- **`evaluate(cache, interval_specs) -> np.ndarray`** evaluates the score on a batch of intervals in one call. The output is a 2D numpy array with one row per interval; the number of columns is either 1 or `n_features` depending on the scorer.

What distinguishes one scorer from another is the **`score_type` tag**, declared via the sklearn-style `__sklearn_tags__` mechanism. The tag tells the detector — and any composition adapter — what kind of score the object produces, and therefore what shape of `interval_specs` it expects:

Detectors expose typed constructor arguments (`cost=`, `change_score=`, `saving=`, …) and reject scorers whose `score_type` tag does not match the slot. Helper predicates `is_cost`, `is_change_score`, `is_saving`, and `is_transient_score` are available in `skchange.new_api.interval_scorers` if you ever need to dispatch on the tag yourself.



To make each kind concrete, the examples below reuse the same `X` defined in the [Change detectors](#change-detectors) section: a univariate series with a change in mean at index 20.



### Cost

A *cost* measures the cost/loss/error of a model fit to a data interval `X[s:e]`. For example, `L2Cost` returns the within-interval sum of squared deviations from the sample mean — small when the interval is well-modelled by a single mean, large when it straddles a change.


In [ ]:
from skchange.new_api.interval_scorers import L2Cost

cost = L2Cost().fit(X)
cache = cost.precompute(X)
cost.evaluate(cache, [[0, 5], [15, 25], [25, 30]])


The cost is much larger for the interval `[15, 25)` than for the other two, because that interval straddles the change point at index 20 and a single-mean model fits poorly there.

The computational bottleneck of most change-detection algorithms is to evaluate a score over a large number of intervals. In Skchange this is solved by splitting work between `fit`/`precompute` (which prepares quantities such as cumulative sums) and `evaluate` (which uses them to score many intervals in a single, often `numba`-accelerated call).



### Change score

A *change score* takes in a start–split–end configuration `(s, k, e)` and measures the degree of change between the two adjacent intervals `X[s:k]` and `X[k:e]`. Change scores can be statistical tests, time-series distances, or any other measure of difference. A classical example is the CUSUM score for a change in mean.


In [ ]:
from skchange.new_api.interval_scorers import CUSUM

score = CUSUM().fit(X)
cache = score.precompute(X)
score.evaluate(cache, [[0, 3, 6], [15, 20, 25], [20, 25, 30]])


Again, the change score is largest for the interval that contains the change point at index 20.

One of the strengths of the interval-scorer abstraction is that **a cost can always be turned into a change score**. The `CostChangeScore` adapter does exactly this:

```python
from skchange.new_api.interval_scorers import CostChangeScore, L2Cost

change_score = CostChangeScore(L2Cost())
```

Internally, `CostChangeScore` evaluates

```python
change_score(s, k, e) = cost(s, e) - (cost(s, k) + cost(k, e))
```

which you can read as *"the cost of the interval without a change point minus the cost of the interval with a single change point"*.

Composition in Skchange is **explicit** — you always pass the exact scorer kind the detector expects. A detector that takes `change_score=` will reject a bare cost; you wrap it in `CostChangeScore` yourself. This keeps detector `repr` and `get_params()` round-trippable: what you passed is exactly what runs.

The library also supports change scores that **cannot** be reduced to costs. This is different from libraries such as `ruptures`. There are quite a few important scores that can not be reduced to costs, including the Mann–Whitney U test, the Kolmogorov–Smirnov test, and scores for sparse change detection in high-dimensional data. Implementing these directly is also often more computationally efficient.



### Saving

A *saving* takes a start–end configuration `(s, e)` and measures the difference in cost between a locally estimated and a globally fixed model parameter over `X[s:e]`. The globally fixed parameter represents the "normal" data behaviour and is user-specified. In practice it can be estimated robustly from the data. Savings are the main type of anomaly score used in Skchange because they are cheap to evaluate over many intervals.

Unlike change scores, a cost in Skchange cannot be used to automatically make a saving because Skchange costs don't have a notion of a fixed parameter.


In [ ]:
import numpy as np

from skchange.new_api.interval_scorers import L2Saving

baseline_mean = float(np.median(X[: X.shape[0] // 2]))
print(f"Baseline mean: {baseline_mean:.3f}")

saving = L2Saving(baseline_mean=baseline_mean).fit(X)
cache = saving.precompute(X)
saving.evaluate(cache, [[0, 5], [15, 25], [20, 30]])


The saving is largest for the last interval, whose mean differs most from the baseline.

### Transient score

A *transient score* (also known as a local anomaly score) receives a start–split1–split2–end configuration `(s, k1, k2, e)` and measures how anomalous the interval `X[k1:k2]` is compared to the local context to its left and right, `X[s:k1]` and `X[k2:e]`. In the literature, such anomalies are sometimes called epidemic changes. Like change scores, **a cost can always be turned into a transient score** via the `CostTransientScore` adapter. Transient scores are typically heavier to compute than savings, because there are many more start–split1–split2–end combinations than start–end ones.


## Penalties